# 02f Three-Feature Machine-Learning Comparison

**Research question.** Can DNN, random forest, or XGBoost improve the day
decision when each receives exactly the same F1/F3/F4 physical evidence?

Every model receives one candidate per day selected by the equal-weight
F1/F3/F4 physical score. The ML model sees only those three feature values and
replaces only the final day decision. Calendar variables, substation identity,
raw load, solar, and F2/F5/F6/F7/F8/F9 are prohibited inputs.

**Inputs:** the 02c equal-selected daily candidate cache and 02e outer physical
predictions.  
**Outputs:** nested ML metrics, selected hyperparameters, two compact tables,
one figure, and a manifest.  
**Expected runtime:** approximately 25-50 minutes; each completed outer fold is
checkpointed locally.

**Terms.** Reverse power flow (RPF) is real power flowing from the distribution network back into the transmission system.


## 1. Imports, Paths, And Declared Model Grids

The DNN is a standardised two-hidden-layer `MLPClassifier`. Random forest and
XGBoost use deterministic seeds and weighted training rows. The grids are small
and fixed in configuration: 4 DNN, 6 random-forest, and 4 XGBoost combinations.
No broad exploratory search is launched.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import COMPACT_FEATURE_COLUMNS  # noqa: E402
from _m9_pbm_plotting import plot_physical_vs_ml  # noqa: E402
from _m9_pbm_validation import (  # noqa: E402
    metric_rows,
    ml_hyperparameter_definitions,
    run_nested_ml_outer_experiment,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "02f_m9_pbm_ml_comparison"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)
CHECKPOINT_DIR = OUTPUT_DIRS["intermediate"] / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SEED = CONFIG["m9_pbm"]["ml_comparison"]["random_seed"]
DEFINITIONS = ml_hyperparameter_definitions(CONFIG)

assert len(DEFINITIONS) == 14
assert set(DEFINITIONS["model"]) == {"dnn", "random_forest", "xgboost"}
display(
    DEFINITIONS[["model", "hyperparameter_id", "parameters_json"]]
)

## 2. Fair-Comparison Input And Decision

Candidate selection remains the equal physical score

$$
W_d^*=\operatorname*{arg\,max}_{W}
\frac{F_1(W)+F_3(W)+F_4(W)}{3}.
$$

The classifier then estimates

$$
p_d=P(RPF_d=1\mid F_1(W_d^*),F_3(W_d^*),F_4(W_d^*)),
$$

and predicts

$$
\widehat{RPF}(d)=\mathbb{1}\{p_d\geq\tau_{ML}\}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $d$ | One substation-day. |
| $W$ | One valid physical candidate window. |
| $W_d^*$ | Equal-F1/F3/F4 candidate selected before ML. |
| $F_1,F_3,F_4$ | Bridge, slope-continuity, and duration features. |
| $p_d$ | Model-estimated probability of an RPF sign-error day. |
| $\tau_{ML}$ | Probability threshold selected from training data only. |

ML cannot move the window boundaries. This isolates whether a nonlinear day
decision adds value beyond the deterministic physical score.

## 3. Load And Validate Equal-Selected Daily Features

The selected-candidate cache must contain exactly three allowed model inputs.
Labels and confidence are retained for nested training and reporting, not as
features. The final physical predictions are loaded separately for the fair
comparison table.

In [ ]:
DAILY_CACHE = (
    PATHS.intermediate
    / "02c_m9_pbm_training_regimes"
    / "compact_equal_daily_candidates.parquet"
)
PHYSICAL_PREDICTIONS = (
    PATHS.intermediate
    / "02e_m9_pbm_weight_optimisation"
    / "nested_outer_predictions.parquet"
)
assert DAILY_CACHE.exists() and PHYSICAL_PREDICTIONS.exists(), "Run 02c and 02e first."

daily = pd.read_parquet(DAILY_CACHE)
alpha = daily.loc[daily["dataset"].eq("alpha")].copy()
beta = daily.loc[daily["dataset"].eq("beta")].copy()
beta_sure = beta.loc[beta["confidence"].eq("sure")].copy()
beta_substations = sorted(beta["substation_id"].unique())
alpha_substations = sorted(alpha["substation_id"].unique())

assert len(COMPACT_FEATURE_COLUMNS) == 3
assert daily[COMPACT_FEATURE_COLUMNS].notna().all().all()
assert len(beta) == CONFIG["datasets"]["expected_substation_days"]["beta"]
display(
    pd.DataFrame(
        [
            {"dataset": "Alpha", "rows": len(alpha), "training_rows": len(alpha)},
            {"dataset": "Beta sure", "rows": len(beta_sure), "training_rows": len(beta_sure)},
            {"dataset": "Beta all", "rows": len(beta), "training_rows": "reporting only"},
        ]
    )
)

## 4. Nested Beta LOSO For Beta-Only And Beta-Plus-Alpha

For each outer Beta substation, hyperparameters are selected by inner LOSO on
the other seven. Each inner model trains on six Beta substations, plus all Alpha
only in the mixed regime. Its probability threshold is selected on those
training rows before the seventh Beta substation is evaluated. The selected
configuration is then fitted on all seven outer-training Beta substations and
predicts the outer substation once.

Beta training uses sure days only. Outer predictions include all days, and
confidence is consulted only afterward for Beta sure/all reporting. Checkpoints
are written after every completed regime-fold pair.

In [ ]:
inner_parts = []
selected_parts = []
decision_parts = []
for regime in ["beta_only", "beta_plus_alpha"]:
    for outer_heldout in beta_substations:
        stem = f"{regime}_{outer_heldout}"
        inner_path = CHECKPOINT_DIR / f"{stem}_inner.csv"
        selected_path = CHECKPOINT_DIR / f"{stem}_selected.csv"
        decision_path = CHECKPOINT_DIR / f"{stem}_decisions.parquet"
        reusable = (
            CONFIG["execution"]["resume_validated_intermediates"]
            and inner_path.exists()
            and selected_path.exists()
            and decision_path.exists()
        )
        if reusable:
            inner_results = pd.read_csv(inner_path)
            selected = pd.read_csv(selected_path)
            decisions = pd.read_parquet(decision_path)
        else:
            beta_training = beta_sure.loc[
                ~beta_sure["substation_id"].eq(outer_heldout)
            ].copy()
            training_pool = (
                beta_training
                if regime == "beta_only"
                else pd.concat([alpha, beta_training], ignore_index=True)
            )
            outer_evaluation = beta.loc[
                beta["substation_id"].eq(outer_heldout)
            ].copy()
            inner_substations = [
                substation for substation in beta_substations
                if substation != outer_heldout
            ]
            inner_results, selected, decisions = run_nested_ml_outer_experiment(
                training_pool,
                outer_evaluation,
                DEFINITIONS,
                feature_columns=COMPACT_FEATURE_COLUMNS,
                inner_fold_substations=inner_substations,
                dataset_balanced=regime == "beta_plus_alpha",
                seed=SEED,
                regime=regime,
                outer_identifier=outer_heldout,
            )
            write_csv(inner_results, inner_path)
            write_csv(selected, selected_path)
            write_parquet(decisions, decision_path)
        inner_parts.append(inner_results)
        selected_parts.append(selected)
        decision_parts.append(decisions)
        print(f"{regime} {outer_heldout}: {'reused' if reusable else 'completed'}", flush=True)

## 5. Alpha-Only Transfer

Alpha-only hyperparameters are selected by leave-one-Alpha-substation-out
validation. The selected model and probability threshold are then fitted using
all Alpha rows and transferred to all Beta substations. No Beta label or
confidence value enters Alpha-only model selection.

In [ ]:
stem = "alpha_only_all_beta"
inner_path = CHECKPOINT_DIR / f"{stem}_inner.csv"
selected_path = CHECKPOINT_DIR / f"{stem}_selected.csv"
decision_path = CHECKPOINT_DIR / f"{stem}_decisions.parquet"
reusable = (
    CONFIG["execution"]["resume_validated_intermediates"]
    and inner_path.exists()
    and selected_path.exists()
    and decision_path.exists()
)
if reusable:
    alpha_inner = pd.read_csv(inner_path)
    alpha_selected = pd.read_csv(selected_path)
    alpha_decisions = pd.read_parquet(decision_path)
else:
    alpha_inner, alpha_selected, alpha_decisions = run_nested_ml_outer_experiment(
        alpha,
        beta,
        DEFINITIONS,
        feature_columns=COMPACT_FEATURE_COLUMNS,
        inner_fold_substations=alpha_substations,
        dataset_balanced=False,
        seed=SEED,
        regime="alpha_only",
        outer_identifier="all_beta",
    )
    write_csv(alpha_inner, inner_path)
    write_csv(alpha_selected, selected_path)
    write_parquet(alpha_decisions, decision_path)
inner_parts.append(alpha_inner)
selected_parts.append(alpha_selected)
decision_parts.append(alpha_decisions)

inner_results = pd.concat(inner_parts, ignore_index=True)
selected_hyperparameters = pd.concat(selected_parts, ignore_index=True)
ml_decisions = pd.concat(decision_parts, ignore_index=True)
assert len(selected_hyperparameters) == 8 * 2 * 3 + 3
assert ml_decisions.groupby(["regime", "model"]).size().eq(len(beta)).all()
display(
    selected_hyperparameters[
        [
            "regime", "outer_identifier", "model", "hyperparameter_id",
            "inner_macro_f1", "outer_threshold"
        ]
    ]
)

## 6. Held-Out Beta Sure And Beta All Metrics

The primary comparison uses pooled held-out Beta-sure precision, recall, and F1.
Macro-substation and per-substation rows are retained. The deterministic
optimised physical model is evaluated from its 02e outer predictions and cannot
be displaced as the final method regardless of ML ranking.

In [ ]:
metric_parts = []
for (regime, model), model_frame in ml_decisions.groupby(
    ["regime", "model"], sort=False
):
    for confidence_scope, evaluation in [
        ("beta_sure", model_frame.loc[model_frame["confidence"].eq("sure")]),
        ("beta_all", model_frame),
    ]:
        rows = metric_rows(evaluation)
        rows.insert(0, "confidence_scope", confidence_scope)
        rows.insert(0, "model", model)
        rows.insert(0, "regime", regime)
        metric_parts.append(rows)
ml_metrics = pd.concat(metric_parts, ignore_index=True)

physical = pd.read_parquet(PHYSICAL_PREDICTIONS)
physical_metric_parts = []
for confidence_scope, evaluation in [
    ("beta_sure", physical.loc[physical["confidence"].eq("sure")]),
    ("beta_all", physical),
]:
    rows = metric_rows(evaluation)
    rows.insert(0, "confidence_scope", confidence_scope)
    rows.insert(0, "model", "m9_pbm_optimised_physical")
    rows.insert(0, "regime", "nested_beta_only")
    physical_metric_parts.append(rows)
physical_metrics = pd.concat(physical_metric_parts, ignore_index=True)

all_metrics = pd.concat([physical_metrics, ml_metrics], ignore_index=True)
comparison = all_metrics.loc[
    all_metrics["confidence_scope"].eq("beta_sure")
    & all_metrics["aggregation"].isin(["pooled", "macro_substation"])
].copy()
by_substation = all_metrics.loc[all_metrics["aggregation"].eq("substation")].copy()

display(
    comparison[
        [
            "regime", "model", "aggregation", "support", "precision", "recall", "f1"
        ]
    ]
)

## 7. Write Results And Comparison Figure

The figure shows pooled Beta-sure results. Each ML label includes its training
regime, while the physical model appears once using the nested outer predictions
from 02e. Hyperparameter definitions and selected thresholds remain available
in CSV; fitted estimators are intentionally not serialized.

In [ ]:
METRICS_PATH = OUTPUT_DIRS["metrics"] / "01_ml_nested_metrics.csv"
HYPERPARAMETER_PATH = OUTPUT_DIRS["metrics"] / "02_selected_hyperparameters.csv"
COMPARISON_PATH = OUTPUT_DIRS["tables"] / "table01_physical_vs_ml.csv"
SUBSTATION_PATH = OUTPUT_DIRS["tables"] / "table02_ml_by_substation.csv"
write_csv(ml_metrics, METRICS_PATH)
write_csv(selected_hyperparameters, HYPERPARAMETER_PATH)
write_csv(comparison, COMPARISON_PATH)
write_csv(by_substation, SUBSTATION_PATH)

model_labels = {
    "m9_pbm_optimised_physical": "Physical / nested Beta",
    "dnn": "DNN",
    "random_forest": "RF",
    "xgboost": "XGB",
}
regime_labels = {
    "beta_only": "Beta",
    "beta_plus_alpha": "Beta + Alpha",
    "alpha_only": "Alpha",
    "nested_beta_only": "nested Beta",
}
figure_data = comparison.loc[comparison["aggregation"].eq("pooled")].copy()
figure_data = figure_data.loc[
    figure_data["model"].eq("m9_pbm_optimised_physical")
    | figure_data["regime"].eq("beta_only")
].copy()
figure_data["display_label"] = [
    model_labels[model]
    if model == "m9_pbm_optimised_physical"
    else f"{model_labels[model]} / {regime_labels[regime]}"
    for model, regime in zip(figure_data["model"], figure_data["regime"], strict=True)
]
FIGURE_PATH = OUTPUT_DIRS["figures"] / "fig01_physical_vs_ml_precision_recall_f1.png"
plot_physical_vs_ml(figure_data, FIGURE_PATH)
display(FIGURE_PATH)

## 8. Interpretation, Leakage Statement, And Manifest

This experiment tests only the final day decision. Candidate generation and
window selection remain the same compact physical process for all models.
Nested folds prevent the outer Beta substation from selecting ML
hyperparameters or thresholds in the Beta-trained regimes. Alpha-only transfer
uses no Beta labels.

The experiment is model-development evidence because Beta informed earlier
feature selection. The deterministic optimised F1/F3/F4 model remains the final
`m9_pbm` method even if an ML row scores higher.

In [ ]:
MANIFEST_OUTPUTS = [
    METRICS_PATH, HYPERPARAMETER_PATH, COMPARISON_PATH, SUBSTATION_PATH, FIGURE_PATH,
]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[PATHS.config, DAILY_CACHE, PHYSICAL_PREDICTIONS],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "hyperparameter_definitions": len(DEFINITIONS),
        "selected_hyperparameter_rows": len(selected_hyperparameters),
        "ml_outer_decisions": len(ml_decisions),
        "ml_metric_rows": len(ml_metrics),
    },
)
manifest["local_intermediates"] = [str(CHECKPOINT_DIR.relative_to(PATHS.article))]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)
inventory = pd.DataFrame({"path": [*MANIFEST_OUTPUTS, MANIFEST_PATH]})
inventory["exists"] = inventory["path"].map(Path.exists)
inventory["bytes"] = inventory["path"].map(lambda path: path.stat().st_size)
display(inventory)
assert inventory["exists"].all() and inventory["bytes"].gt(0).all()

## Fast Figure-Only Rerender

Run this cell after the lightweight setup cell whenever only the publication
figures need to change. It reads persisted results, refreshes validated
figure-source caches, and does not repeat candidate generation, fitting, or
evaluation.

In [ ]:
from _cached_figure_rendering import render_notebook_figures

RENDER_ONLY = True
if RENDER_ONLY:
    RENDERED_FIGURES = render_notebook_figures(
        ARTICLE_ROOT,
        '02f_m9_pbm_ml_comparison',
        refresh_sources=True,
    )
    display(pd.Series([str(path) for path in RENDERED_FIGURES], name="rendered_figure"))